# TP Inteligencia Artificial 2026 – Decision Tree vs Random Forest

Comparar un **Decision Tree implementado por nosotros** (basado en C4.5) con **Random Forest** (scikit-learn) en clasificación.

**Integrantes:** _completar_  
**Dataset:** UCI Heart Disease (Cleveland, Hungarian, Switzerland y VA Long Beach), en `data/`.  
**Librerías permitidas:** numpy, pandas, graphviz, ipywidgets, scikit-learn.

## Panel de control

**Cómo usarlo**
1. Ejecutá todo el notebook una vez: *Entorno de ejecución → Ejecutar todo* (carga los datos y define los modelos; tarda unos segundos).
2. Volvé a este panel, elegí la configuración en las pestañas **Datos**, **Decision Tree** y **Random Forest**.
3. Elegí qué modelo correr y presioná **▶ Ejecutar**. Los resultados aparecen debajo del panel y cada ejecución reemplaza a la anterior.

Más abajo en el notebook están el análisis de los datos y el código de los dos algoritmos.

In [ ]:
#@title Panel de control (ejecutar con "Ejecutar todo" y usar los botones)
import os, time, warnings
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore", message="Skipping features without any observed values")

def ejecutar_experimento(modo_clases, test_size, semilla, modelo, dt_cfg, rf_cfg, bases=None):
    """Entrena y evalúa según la configuración. Devuelve una lista de dicts con resultados."""
    y_full = df[TARGET].values
    if modo_clases == "binaria" and np.issubdtype(y_full.dtype, np.number):
        y_full = (y_full > 0).astype(int)                    # 0 = sano, 1 = enfermo
    mask = np.ones(len(df), dtype=bool) if bases is None else df["origen"].isin(bases).values
    X_sel, y_full = X.loc[mask], y_full[mask]               # solo las bases elegidas
    Xtr, Xte, ytr, yte = train_test_split(X_sel, y_full, test_size=test_size,
                                          stratify=y_full, random_state=semilla)
    etiquetas = np.unique(y_full)
    modelos = []
    if modelo in ("DT propio", "Ambos"):
        modelos.append(("DT propio", dt_cfg, ArbolDecision(categoricos=CATEGORICOS, **dt_cfg)))
    if modelo in ("Random Forest", "Ambos"):
        modelos.append(("Random Forest", rf_cfg, crear_rf(**rf_cfg)))
    salidas = []
    for nombre, cfg, m in modelos:
        t0 = time.perf_counter(); m.fit(Xtr, ytr); t_fit = time.perf_counter() - t0
        t0 = time.perf_counter(); pred = m.predict(Xte); t_pred = time.perf_counter() - t0
        fila = {"modelo": nombre, "clases": modo_clases,
                "bases": "todas" if bases is None else "+".join(bases), "filas": int(mask.sum()),
                "test %": round(100 * test_size), "semilla": semilla,
                **metricas(yte, pred), "t_entren (s)": t_fit, "t_pred (s)": t_pred, "params": str(cfg)}
        if nombre == "DT propio":
            fila["hojas"], fila["prof"] = m.n_hojas(), m.profundidad()
        cm = pd.DataFrame(confusion_matrix(yte, pred, labels=etiquetas),
                          index=[f"real {e}" for e in etiquetas],
                          columns=[f"pred {e}" for e in etiquetas])
        rep = pd.DataFrame(classification_report(yte, pred, labels=etiquetas,
                                                 output_dict=True, zero_division=0)).T
        rep = rep.loc[[str(e) for e in etiquetas], ["precision", "recall", "f1-score", "support"]]
        salidas.append({"fila": fila, "cm": cm, "rep": rep, "modelo": m})
    return salidas


# ---------------- presentación de resultados (HTML que funciona en tema claro y oscuro) ----------------
_CSS = """<style>
.tp{font-family:inherit;line-height:1.45}
.tp-h{font-size:1.1em;font-weight:600;margin:20px 0 8px 0}
.tp-info{opacity:.85;margin:4px 0 12px 0}
.tp-small{opacity:.75;font-size:.9em}
.tp-flex{display:flex;flex-wrap:wrap;gap:16px;align-items:flex-start}
.tp-card{border:1px solid rgba(128,128,128,.45);border-radius:10px;padding:12px 18px;min-width:260px}
.tp-card.best{border:2px solid #3fb950}
.tp-big{font-size:2em;font-weight:700;line-height:1.1}
.tp-row{display:flex;gap:26px;margin:6px 0}
table.tp-t{border-collapse:collapse;margin:4px 0 10px 0}
table.tp-t th,table.tp-t td{border:1px solid rgba(128,128,128,.35);padding:4px 11px;text-align:center}
table.tp-t th{background:rgba(128,128,128,.18);font-weight:600}
.tp-scroll{overflow-x:auto;max-width:100%}
</style>"""
_NOMBRE_CLASES = {"multiclase": "5 clases", "binaria": "2 clases (sano / enfermo)"}


def _tarjeta(f, mejor):
    extra = f"<div class='tp-small'>Árbol: {f['hojas']} hojas, profundidad {f['prof']}</div>" if "hojas" in f else ""
    marca = " · mejor F1" if mejor else ""
    return (f"<div class='tp-card{' best' if mejor else ''}'>"
            f"<div class='tp-small'>{f['modelo']}{marca}</div>"
            f"<div class='tp-row'><div><div class='tp-big'>{f['accuracy']:.1%}</div><div class='tp-small'>accuracy</div></div>"
            f"<div><div class='tp-big'>{f['f1']:.1%}</div><div class='tp-small'>F1 macro</div></div></div>"
            f"<div class='tp-small'>precision {f['precision']:.1%} · recall {f['recall']:.1%}</div>"
            f"<div class='tp-small'>entrenar {f['t_entren (s)']:.2f} s · predecir {f['t_pred (s)']:.3f} s</div>"
            f"{extra}</div>")


def _html_cm(cm):
    tot = cm.values.sum(axis=1)
    h = "<table class='tp-t'><tr><th></th>" + "".join(f"<th>{c}</th>" for c in cm.columns) + "</tr>"
    for i, (nombre, fila) in enumerate(zip(cm.index, cm.values)):
        h += f"<tr><th>{nombre}</th>"
        for j, v in enumerate(fila):
            frac = v / tot[i] if tot[i] else 0
            color = "46,160,67" if i == j else "219,68,55"      # verde = acierto, rojo = error
            alfa = round(0.12 + 0.65 * frac, 2) if v > 0 else 0
            peso = 700 if i == j else 400
            h += f"<td style='background:rgba({color},{alfa});font-weight:{peso}'>{v}</td>"
        h += "</tr>"
    return h + "</table>"


def _html_rep(rep):
    h = "<table class='tp-t'><tr><th>clase</th><th>precision</th><th>recall</th><th>f1</th><th>casos</th></tr>"
    for c, r in rep.iterrows():
        alfa = round(0.1 + 0.6 * r["f1-score"], 2)
        h += (f"<tr><th>{c}</th><td>{r['precision']:.2f}</td><td>{r['recall']:.2f}</td>"
              f"<td style='background:rgba(46,160,67,{alfa})'>{r['f1-score']:.2f}</td><td>{int(r['support'])}</td></tr>")
    return h + "</table>"


def _html_resultados(res):
    f0 = res[0]["fila"]
    h = _CSS + "<div class='tp'>"
    h += (f"<div class='tp-info'>Bases: <b>{f0['bases']}</b> ({f0['filas']} pacientes) · "
          f"{_NOMBRE_CLASES[f0['clases']]} · {f0['test %']} % para prueba · semilla {f0['semilla']}</div>")
    mejor = max(res, key=lambda r: r["fila"]["f1"]) if len(res) > 1 else None
    h += "<div class='tp-flex'>" + "".join(_tarjeta(r["fila"], r is mejor) for r in res) + "</div>"
    h += ("<div class='tp-small' style='margin-top:6px'>Resultados sobre los datos de prueba. "
          "accuracy = % de aciertos. F1, precision y recall son el promedio de todas las clases (macro): "
          "una clase chica pesa igual que una grande.</div>")
    for r in res:
        h += f"<div class='tp-h'>{r['fila']['modelo']}: detalle</div><div class='tp-flex'>"
        h += ("<div><div class='tp-small'>Matriz de confusión (filas = clase real, columnas = predicha; "
              "verde = acierto, rojo = error)</div>" + _html_cm(r["cm"]) + "</div>")
        h += "<div><div class='tp-small'>Métricas por clase</div>" + _html_rep(r["rep"]) + "</div></div>"
    return h + "</div>"


def dibujar_arbol(modelo, niveles):
    """Dibuja el árbol aprendido con graphviz (en Colab ya viene instalado)."""
    leyenda = "".join(
        f"<span style='background:{ArbolDecision.COLORES[i % 5]};color:#000;padding:2px 9px;margin-right:6px;"
        f"border-radius:4px'>clase {c}</span>" for i, c in enumerate(modelo.clases_))
    display(HTML(_CSS + "<div class='tp'><div class='tp-h'>Árbol aprendido (primeros "
                 f"{niveles} niveles)</div><div class='tp-small'>Cada caja: atributo de división, n = ejemplos de "
                 "entrenamiento que llegaron, [..] = ejemplos por clase. Colores de las hojas: "
                 f"{leyenda}</div></div>"))
    try:
        import graphviz
        svg = graphviz.Source(modelo.a_dot(niveles)).pipe(format="svg").decode("utf-8")
        display(HTML("<div style='background:#fff;display:inline-block;padding:6px;border-radius:8px'>"
                     + svg + "</div>"))
    except Exception as e:                       # sin graphviz: se muestra en texto
        print(f"(No se pudo dibujar con graphviz: {type(e).__name__}. Se muestra en texto.)")
        modelo.imprimir(max_prof=niveles)


# ---------------- análisis de las bases de datos ----------------
_NIVELES = {"alto": ("ALTO", "#e5534b"), "medio": ("MEDIO", "#f0a500"), "info": ("INFO", "#4c9be8")}


def _avisos_datos(claves, o, y, numerico, miss):
    """Lista de (nivel, texto) con los problemas detectados en los datos cargados."""
    av = []
    n_por_base = {b: int((o == b).sum()) for b in claves}

    # 1) clases que no aparecen en una base
    if numerico:
        todas = sorted(y.unique())
        for b in claves:
            ausentes = [c for c in todas if int((y[o == b] == c).sum()) == 0]
            if len(ausentes) >= 2:
                txt = (f"<b>{b}</b> no tiene ningún paciente de las clases {', '.join(str(c) for c in ausentes)}. "
                       "Con 5 clases, esa base solo aporta sanos y un único nivel de enfermedad.")
                if b == "hungarian" and set(ausentes) >= {2, 3, 4}:
                    txt += (" La documentación (heart-disease.names) indica que Hungarian debería tener 37, 26, 28 y 15 "
                            "casos en las clases 1 a 4: este archivo (<code>processed</code>) parece tener los 4 niveles "
                            "de enfermedad agrupados en la clase 1. El archivo <code>reprocessed.hungarian.data</code> "
                            "conserva las 5 clases.")
                av.append(("alto", txt))

    # 2) ceros imposibles
    for b in claves:
        m = o == b
        c0 = int((df.loc[m, "chol"] == 0).sum())
        if c0:
            pct = 100 * c0 / n_por_base[b]
            av.append(("alto" if pct >= 30 else "medio",
                       f"<b>{b}</b>: colesterol (<code>chol</code>) igual a 0 en {c0} de {n_por_base[b]} pacientes "
                       f"({pct:.0f} %). Es un valor imposible: se interpreta como faltante."
                       + (" La columna entera es inválida en esta base." if c0 == n_por_base[b] else "")))
        p0 = int((df.loc[m, "trestbps"] == 0).sum())
        if p0:
            av.append(("medio", f"<b>{b}</b>: presión en reposo (<code>trestbps</code>) igual a 0 en {p0} paciente(s): valor imposible."))

    # 3) atributos con muchos faltantes o con un solo valor observado
    for b in claves:
        m = o == b
        pct = miss[m].mean() * 100
        muchos = [a for a in FEATURES if pct[a] >= 50]
        if muchos:
            det = ", ".join(f"{a} ({pct[a]:.0f} %)" for a in muchos)
            casi = any(pct[a] >= 90 for a in muchos)
            av.append(("alto" if casi else "medio", f"<b>{b}</b>: faltan más de la mitad de los datos de {det}."))
        for a in FEATURES:
            vals = X.loc[m, a].dropna()
            if 0 < len(vals) and vals.nunique() == 1:
                av.append(("medio", f"<b>{b}</b>: <code>{a}</code> tiene un único valor observado "
                                    f"({vals.iloc[0]:g}, en {len(vals)} pacientes): no aporta información."))
        n5 = int((miss[m].sum(axis=1) >= 5).sum())
        if n5 / n_por_base[b] >= 0.10:
            av.append(("medio", f"<b>{b}</b>: {n5} pacientes ({100 * n5 / n_por_base[b]:.0f} %) con 5 o más "
                                f"de los {len(FEATURES)} atributos faltantes."))

    # 4) desbalance y sesgo de sexo por base
    if numerico:
        for b in claves:
            enf = 100 * (y[o == b] > 0).mean()
            if enf > 75 or enf < 25:
                av.append(("medio", f"<b>{b}</b>: {enf:.0f} % de pacientes enfermos. En sano vs. enfermo, responder "
                                    f"siempre la clase más frecuente ya acertaría {max(enf, 100 - enf):.0f} %: mirar F1 y matriz de confusión."))
        tasas = {b: 100 * (y[o == b] > 0).mean() for b in claves}
        if max(tasas.values()) - min(tasas.values()) > 30:
            av.append(("alto", "El porcentaje de enfermos cambia mucho entre bases "
                               f"({min(tasas.values()):.0f} % a {max(tasas.values()):.0f} %) y cada base tiene sus propios "
                               "patrones de faltantes. Un modelo entrenado con todas puede aprender «de qué hospital viene» "
                               "el paciente en lugar de la enfermedad: conviene comparar contra resultados con una sola base."))
    for b in claves:
        h = 100 * df.loc[o == b, "sex"].mean()
        if h > 85:
            av.append(("info", f"<b>{b}</b>: {h:.0f} % de los pacientes son hombres (muestra poco representativa de mujeres)."))

    # 5) clases con pocos casos
    if numerico:
        for c, n in y.value_counts().sort_index().items():
            if n < 40:
                av.append(("medio", f"La clase {c} tiene solo {n} pacientes en total: con 30 % de prueba quedan "
                                    f"~{round(n * 0.3)}, y sus métricas son muy inestables."))

    # 6) valores sospechosos y duplicados
    neg = df[df["oldpeak"] < 0]
    if len(neg):
        det = ", ".join(f"{b} ({n})" for b, n in neg["origen"].value_counts().items())
        av.append(("info", f"<code>oldpeak</code> negativo en {len(neg)} pacientes: {det}. Puede ser válido, conviene revisarlo."))
    for col, lim, txt in [("chol", 500, "colesterol mayor a 500"), ("thalach", 70, "frecuencia máxima menor a 70")]:
        n = int((df[col] > lim).sum()) if col == "chol" else int((df[col] < lim).sum())
        if n:
            av.append(("info", f"{n} pacientes con {txt} (valores extremos, posibles errores de carga)."))
    dup = int(df[FEATURES + [TARGET]].duplicated().sum())
    if dup:
        av.append(("info", f"{dup} filas duplicadas (mismos valores en todos los atributos y la clase)."))
    tot = miss.mean() * 100
    top = ", ".join(f"{a} ({tot[a]:.0f} %)" for a in tot.sort_values(ascending=False).index[:3] if tot[a] >= 30)
    if top:
        av.append(("info", f"Atributos más incompletos en el total: {top}."))
    orden = {"alto": 0, "medio": 1, "info": 2}
    return sorted(av, key=lambda a: orden[a[0]])


def _html_analisis():
    claves = [k for k, _ in BASES]
    o, y = df["origen"], df[TARGET]
    numerico = np.issubdtype(y.dtype, np.number)
    miss = X.isna()                                   # incluye los ceros inválidos de chol y trestbps
    rojo = "background:rgba(219,68,55,.35)"

    clases = sorted(y.unique()) if numerico else []
    cab = ["Base", "Pacientes"] + (["% enfermos"] if numerico else []) + ["% hombres", "Edad media"] \
        + [f"clase {c}" for c in clases] + ["Filas con ≥5 faltantes"]
    res = "<table class='tp-t'><tr>" + "".join(f"<th>{c}</th>" for c in cab) + "</tr>"
    for b in claves:
        m = o == b
        cel = [f"<th>{b}</th>", f"<td>{int(m.sum())}</td>"]
        if numerico:
            cel.append(f"<td>{100 * (y[m] > 0).mean():.0f} %</td>")
        cel += [f"<td>{100 * df.loc[m, 'sex'].mean():.0f} %</td>", f"<td>{df.loc[m, 'age'].mean():.1f}</td>"]
        for c in clases:
            n = int((y[m] == c).sum())
            cel.append(f"<td style='{rojo if n == 0 else ''}'>{n}</td>")
        n5 = int((miss[m].sum(axis=1) >= 5).sum())
        cel.append(f"<td style='{rojo if n5 / m.sum() >= .1 else ''}'>{n5}</td>")
        res += "<tr>" + "".join(cel) + "</tr>"
    res += "</table>"

    hm = "<table class='tp-t'><tr><th>Atributo</th>" + "".join(f"<th>{b}</th>" for b in claves) + "<th>Total</th></tr>"
    pct = {b: miss[o == b].mean() * 100 for b in claves}
    tot = miss.mean() * 100
    for a in FEATURES:
        hm += f"<tr><th>{a}</th>"
        for v in [pct[b][a] for b in claves] + [tot[a]]:
            alfa = round(0.1 + 0.65 * v / 100, 2)
            hm += f"<td style='background:rgba(219,68,55,{alfa})'>{v:.0f} %</td>" if v > 0 else "<td>0 %</td>"
        hm += "</tr>"
    hm += "</table>"

    av = ""
    for nivel, txt in _avisos_datos(claves, o, y, numerico, miss):
        etiqueta, color = _NIVELES[nivel]
        av += (f"<div style='border-left:5px solid {color};padding:6px 12px;margin:6px 0'>"
               f"<span style='color:{color};font-weight:700;font-size:.85em'>{etiqueta}</span> &nbsp;{txt}</div>")

    return (_CSS + "<div class='tp'>"
            "<div class='tp-h'>Avisos sobre los datos</div>" + av +
            "<div class='tp-h'>Resumen por base</div>"
            "<div class='tp-small'>En rojo: clases que no existen en la base y bases con muchas filas incompletas.</div>"
            + res +
            "<div class='tp-h'>Datos faltantes por atributo y base</div>"
            "<div class='tp-small'>Porcentaje de pacientes sin dato (incluye los ceros imposibles de "
            "<code>chol</code> y <code>trestbps</code>). Más rojo = más faltante.</div>"
            + hm + "</div>")


# ---------------- controles ----------------
_est = {"description_width": "210px"}
_lay = widgets.Layout(width="540px")


def _ayuda(texto):
    return widgets.HTML(f"<div style='opacity:.85;margin:4px 0 12px 0;max-width:640px;line-height:1.45'>{texto}</div>")


def _slider(valor, mn, mx, desc, tip, disabled=False):
    return widgets.IntSlider(value=valor, min=mn, max=mx, description=desc, style=_est, layout=_lay,
                             description_tooltip=tip, disabled=disabled, continuous_update=False)


# --- Datos
BASES = [("cleveland", "Cleveland (EE.UU., 303 pacientes)"),
         ("hungarian", "Hungarian (Hungría, 294 pacientes)"),
         ("switzerland", "Switzerland (Suiza, 123 pacientes)"),
         ("va", "VA Long Beach (EE.UU., 200 pacientes)")]
w_bases = {k: widgets.Checkbox(value=True, description=d, indent=False, layout=_lay) for k, d in BASES}
w_clases = widgets.Dropdown(
    options=[("5 clases: sano + 4 niveles de enfermedad", "multiclase"),
             ("2 clases: sano vs. enfermo", "binaria")],
    value="multiclase", description="Clases a predecir", style=_est, layout=_lay,
    description_tooltip="Con 5 clases el problema es más difícil; con 2 se agrupan los 4 niveles de enfermedad.")
w_test = _slider(30, 10, 50, "% de datos para prueba",
                 "Porcentaje de pacientes que NO se usa para entrenar y sirve para medir el modelo.")
w_semilla = widgets.IntText(value=42, description="Semilla aleatoria", style=_est, layout=_lay,
                            description_tooltip="Mismo número = mismos resultados. Cambiarlo cambia el reparto al azar.")

# --- Decision Tree
w_dt_crit = widgets.Dropdown(
    options=[("Razón de ganancia (C4.5)", "razon_ganancia"), ("Ganancia de información", "ganancia")],
    description="Criterio de división", style=_est, layout=_lay,
    description_tooltip="Cómo elige el árbol la pregunta de cada nodo.")
w_dt_lim = widgets.Checkbox(value=False, description="Limitar la profundidad del árbol",
                            indent=False, layout=_lay)
w_dt_prof = _slider(5, 1, 30, "Profundidad máxima",
                    "Cantidad máxima de preguntas seguidas. Menos profundidad = árbol más simple.", disabled=True)
w_dt_split = _slider(2, 2, 50, "Mín. ejemplos para dividir",
                     "Un nodo con menos ejemplos que esto ya no se divide.")
w_dt_hoja = _slider(1, 1, 30, "Mín. ejemplos por hoja",
                    "Cada hoja final debe tener al menos esta cantidad de ejemplos.")
w_dt_graf = widgets.Checkbox(value=True, description="Dibujar el árbol aprendido",
                             indent=False, layout=_lay)
w_dt_niv = _slider(3, 1, 8, "Niveles a dibujar",
                   "Un árbol completo tiene demasiados nodos para leerlo: se dibujan solo los primeros niveles.")

# --- Random Forest
w_rf_n = _slider(100, 1, 500, "Cantidad de árboles", "Más árboles = resultados más estables pero más lento.")
w_rf_lim = widgets.Checkbox(value=False, description="Limitar la profundidad de cada árbol",
                            indent=False, layout=_lay)
w_rf_prof = _slider(5, 1, 30, "Profundidad máxima",
                    "Cantidad máxima de preguntas seguidas en cada árbol.", disabled=True)
w_rf_crit = widgets.Dropdown(options=[("Entropía", "entropy"), ("Gini", "gini")],
                             description="Criterio de división", style=_est, layout=_lay,
                             description_tooltip="Medida de impureza que usa cada árbol para dividir.")
w_rf_feat = widgets.Dropdown(
    options=[("Raíz cuadrada de N (recomendado)", "sqrt"), ("Logaritmo de N", "log2"), ("Todos los atributos", None)],
    description="Atributos por división", style=_est, layout=_lay,
    description_tooltip="Cuántos atributos al azar mira cada árbol en cada división (N = total de atributos).")

w_dt_lim.observe(lambda c: setattr(w_dt_prof, "disabled", not c["new"]), names="value")
w_rf_lim.observe(lambda c: setattr(w_rf_prof, "disabled", not c["new"]), names="value")

# --- Ejecución
w_modelo = widgets.ToggleButtons(
    options=[("Decision Tree", "DT propio"), ("Random Forest", "Random Forest"), ("Ambos (comparar)", "Ambos")],
    value="Ambos", description="Modelo a ejecutar", style={"description_width": "150px", "button_width": "170px"})
_bl = widgets.Layout(width="190px", height="38px", margin="0 10px 8px 0")
b_run = widgets.Button(description="Ejecutar", button_style="success", icon="play", layout=_bl)
b_reset = widgets.Button(description="Valores por defecto", icon="undo", layout=_bl)
salida = widgets.Output()
b_analizar = widgets.Button(description="Analizar las bases de datos", button_style="info", icon="search",
                            layout=widgets.Layout(width="260px", height="38px", margin="4px 0 8px 0"))
salida_analisis = widgets.Output()


def _aviso(texto):
    display(HTML(f"<div style='border-left:4px solid #f0a500;padding:6px 12px;margin:6px 0'>{texto}</div>"))


def al_ejecutar(_):
    with salida:
        clear_output(wait=True)
        if "ArbolDecision" not in globals() or "X" not in globals() or "crear_rf" not in globals():
            _aviso("Primero ejecutá todo el notebook (<i>Entorno de ejecución → Ejecutar todo</i>) y volvé a probar.")
            return
        bases = [k for k, w in w_bases.items() if w.value]
        if not bases:
            _aviso("Elegí al menos una base de datos en la pestaña <b>1. Datos</b>.")
            return
        dt_cfg = dict(criterio=w_dt_crit.value,
                      max_profundidad=w_dt_prof.value if w_dt_lim.value else None,
                      min_muestras_split=w_dt_split.value, min_muestras_hoja=w_dt_hoja.value)
        rf_cfg = dict(n_arboles=w_rf_n.value,
                      max_profundidad=w_rf_prof.value if w_rf_lim.value else None,
                      criterio=w_rf_crit.value, max_atributos=w_rf_feat.value)
        display(HTML("<i>Ejecutando...</i>"))
        try:
            res = ejecutar_experimento(w_clases.value, w_test.value / 100, w_semilla.value,
                                       w_modelo.value, dt_cfg, rf_cfg,
                                       bases=None if len(bases) == len(BASES) else bases)
        except ValueError as e:
            clear_output(wait=True)
            _aviso("No se pudo armar la partición con esas bases (alguna clase tiene muy pocos casos). "
                   "Probá agregar otra base, usar 2 clases o bajar el % de prueba.<br>"
                   f"<span style='opacity:.75'>Detalle: {e}</span>")
            return
        clear_output(wait=True)
        display(HTML(_html_resultados(res)))
        for r in res:
            if r["fila"]["modelo"] == "DT propio" and w_dt_graf.value:
                dibujar_arbol(r["modelo"], w_dt_niv.value)


def al_restaurar(_):
    for w, v in [*[(w, True) for w in w_bases.values()], (w_clases, "multiclase"), (w_test, 30), (w_semilla, 42),
                 (w_dt_crit, "razon_ganancia"), (w_dt_lim, False), (w_dt_prof, 5), (w_dt_split, 2),
                 (w_dt_hoja, 1), (w_dt_graf, True), (w_dt_niv, 3), (w_rf_n, 100), (w_rf_lim, False),
                 (w_rf_prof, 5), (w_rf_crit, "entropy"), (w_rf_feat, "sqrt"), (w_modelo, "Ambos")]:
        w.value = v


def al_analizar(_):
    with salida_analisis:
        clear_output(wait=True)
        if "X" not in globals() or "df" not in globals():
            _aviso("Primero ejecutá todo el notebook (<i>Entorno de ejecución → Ejecutar todo</i>) y volvé a probar.")
            return
        display(HTML(_html_analisis()))


b_analizar.on_click(al_analizar)
b_run.on_click(al_ejecutar)
b_reset.on_click(al_restaurar)

tab_datos = widgets.VBox([
    _ayuda("Define qué se predice y cómo se reparten los pacientes entre <b>entrenamiento</b> (el modelo aprende) "
           "y <b>prueba</b> (se mide qué tan bien acierta con casos que nunca vio)."),
    widgets.HTML("<b>Bases de datos a usar</b> <span style='opacity:.8'>(cada una viene de un hospital distinto; "
                 "probá combinaciones para ver cuál aporta o perjudica)</span>"),
    *w_bases.values(),
    widgets.HTML("<div style='height:8px'></div>"),
    w_clases, w_test, w_semilla])
tab_dt = widgets.VBox([
    _ayuda("<b>Árbol de decisión propio (C4.5).</b> Hace una serie de preguntas sobre los atributos "
           "(por ej. «edad ≤ 54») hasta llegar a una clase. Un árbol muy profundo memoriza los datos de "
           "entrenamiento y falla con datos nuevos: limitar la profundidad lo simplifica."),
    w_dt_crit, w_dt_lim, w_dt_prof, w_dt_split, w_dt_hoja, w_dt_graf, w_dt_niv])
tab_rf = widgets.VBox([
    _ayuda("<b>Random Forest (scikit-learn).</b> Entrena muchos árboles, cada uno con una muestra al azar de "
           "los datos y de los atributos, y la clase final sale de la votación de todos."),
    w_rf_n, w_rf_lim, w_rf_prof, w_rf_crit, w_rf_feat])

tab_analisis = widgets.VBox([
    _ayuda("Revisa la <b>calidad de las 4 bases</b>: clases que faltan, datos imposibles, atributos casi vacíos, "
           "desbalance y sesgos. Sirve para entender por qué los resultados cambian según las bases elegidas."),
    b_analizar, salida_analisis])

tabs = widgets.Tab(children=[tab_datos, tab_dt, tab_rf, tab_analisis])
for i, t in enumerate(["1. Datos", "2. Decision Tree", "3. Random Forest", "4. Análisis de datos"]):
    tabs.set_title(i, t)

panel = widgets.VBox([
    tabs,
    widgets.HTML("<div style='height:10px'></div>"),
    w_modelo,
    widgets.HBox([b_run, b_reset], layout=widgets.Layout(flex_flow="row wrap", margin="6px 0 0 0")),
])
display(panel, salida)

## 1. Entorno

In [ ]:
import random
import numpy as np
import pandas as pd
import sklearn
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__} | ipywidgets {widgets.__version__}")

## 2. Datos

Dataset **UCI Heart Disease**: 4 archivos `processed.*.data` (sin encabezado, `?` = valor faltante) con 13 atributos + `num` (0 = sin enfermedad, 1–4 = grados de enfermedad).
Se unen en un solo DataFrame, se agrega la columna `origen` (solo informativa, **no** se usa como atributo) y por defecto se **mantienen las clases originales** de `num` (0–4, problema multiclase). Con `BINARIZAR = True` se reduce a enfermo/sano.

En Colab: subir la carpeta `data/` a la sesión (o clonar el repo) y ajustar `DIR_DATOS` si hace falta.

In [ ]:
import os, subprocess

# En Colab la carpeta data/ no existe: se clona el repositorio una sola vez.
if not (os.path.isdir("data") or os.path.isdir("../data")):
    subprocess.run(["git", "clone", "https://github.com/amil-nicolas21/Trabajo-Practico-IA.git"], check=True)
    os.chdir("Trabajo-Practico-IA")
print("Directorio de trabajo:", os.getcwd())

In [ ]:
import os

DIR_DATOS = "../data" if os.path.isdir("../data") else "data"   # notebook en src/ o en la raíz
ARCHIVOS = ["cleveland", "hungarian", "switzerland", "va"]
COLUMNAS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
BINARIZAR = False      # False: clase original num (0-4), sin binarizar; True: enfermo (num>0) vs sano
TARGET = "target"


def cargar_dataset(dir_datos=DIR_DATOS, binarizar=BINARIZAR):
    """Une los 4 archivos de Heart Disease. Devuelve (df, nombre_de_la_clase)."""
    partes = []
    for nombre in ARCHIVOS:
        d = pd.read_csv(f"{dir_datos}/processed.{nombre}.data", header=None,
                        names=COLUMNAS, na_values="?")
        d["origen"] = nombre
        partes.append(d)
    df = pd.concat(partes, ignore_index=True)
    df[TARGET] = (df.pop("num") > 0).map({False: "sano", True: "enfermo"}) if binarizar else df.pop("num").astype(int)
    return df, TARGET


df, TARGET = cargar_dataset()
FEATURES = [c for c in df.columns if c not in (TARGET, "origen")]
print(f"Dataset: {df.shape[0]} filas x {len(FEATURES)} atributos | objetivo: '{TARGET}'")
df.head()

## 3. Análisis exploratorio (EDA)

In [ ]:
# Tipos de atributos
numericos = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
categoricos = [c for c in FEATURES if c not in numericos]
print(f"Numéricos: {len(numericos)} | Categóricos: {len(categoricos)}")
if categoricos:
    print("Categóricos:", categoricos)

# Valores faltantes
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0]
print()
print("Valores faltantes:", "ninguno" if faltantes.empty else "")
if not faltantes.empty:
    display(pd.DataFrame({"faltantes": faltantes, "%": (100 * faltantes / len(df)).round(2)}))

# Balance de clases
balance = df[TARGET].value_counts()
display(pd.DataFrame({"cantidad": balance, "%": (100 * balance / len(df)).round(2)}))

In [ ]:
# Resumen estadístico de los atributos numéricos
df[numericos].describe().T.round(3)

In [ ]:
# Distribución de un atributo numérico (histograma en texto, sin librerías gráficas)
def histograma(col, bins=10, ancho=40):
    cuentas, bordes = np.histogram(df[col].dropna(), bins=bins)
    for c, a, b in zip(cuentas, bordes[:-1], bordes[1:]):
        print(f"{a:10.3f} – {b:10.3f} | {'#' * int(ancho * c / cuentas.max()):<{ancho}} {c}")

histograma(numericos[0])

In [ ]:
# Pares de atributos numéricos más correlacionados (|r| más alto)
corr = df[numericos].corr().abs()
pares = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(ascending=False)
pares.head(10).rename("|correlación|").to_frame().round(3)

**Conclusiones del EDA** _(completar)_. Puntos a analizar, ya detectados al cargar los datos:
- Muchos faltantes: `ca` ≈ 66 %, `thal` ≈ 53 %, `slope` ≈ 34 % → definir cómo los trata el DT (C4.5 maneja faltantes) y RF (imputar).
- `chol = 0` en 172 filas (todas de Switzerland y VA) es un valor inválido: tratarlo como faltante.
- `cp`, `restecg`, `slope`, `thal`, `sex`, `fbs`, `exang` son categóricos codificados como números: no tratarlos como continuos.
- Balance de clases (binarizada vs 5 clases).

## 3b. Preparación para los modelos

- `chol = 0` y `trestbps = 0` son valores imposibles → se pasan a faltante (`NaN`).
- Atributos **categóricos** (códigos sin orden): `sex, cp, fbs, restecg, exang, slope, thal`. El resto se trata como numérico (`ca` incluido).
- Partición **estratificada** 70 % entrenamiento / 30 % prueba, para conservar la proporción de las 5 clases (la clase 4 es muy chica).
- El DT propio maneja faltantes por sí mismo; para RF se imputan con la mediana (dentro del *pipeline*, solo con datos de entrenamiento).

In [ ]:
CATEGORICOS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

X = df[FEATURES].copy()
for col in ["chol", "trestbps"]:
    X.loc[X[col] == 0, col] = np.nan          # 0 no es un valor válido
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)
print(f"Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}")
print("Clases en prueba:", dict(zip(*np.unique(y_test, return_counts=True))))

## 4. Decision Tree (implementación propia, tipo C4.5)

Características de esta versión inicial:
- **Criterio de división:** razón de ganancia (*gain ratio*) de C4.5, o ganancia de información simple (`criterio="ganancia"`). Como en C4.5, solo compiten los atributos cuya ganancia es al menos el promedio.
- **Atributos numéricos:** división binaria `<= umbral` / `> umbral`; el umbral es el punto medio entre valores consecutivos que maximiza la ganancia.
- **Atributos categóricos:** una rama por cada valor.
- **Faltantes:** la ganancia se calcula solo con las filas conocidas y se pondera por su proporción; al construir y al predecir, las filas con valor faltante (o categoría no vista) van a la rama más poblada.
- **Criterios de corte:** nodo puro, `max_profundidad`, `min_muestras_split`, `min_muestras_hoja` o ninguna ganancia positiva.
- **Pendiente:** poda (pessimistic error pruning de C4.5).

In [ ]:
def _entropia(C):
    """Entropía en bits de cada fila de una matriz de conteos por clase."""
    C = np.asarray(C, dtype=float)
    n = C.sum(axis=-1, keepdims=True)
    p = C / np.where(n == 0, 1, n)
    return -(p * np.log2(np.where(p > 0, p, 1))).sum(axis=-1)


class Nodo:
    """Nodo del árbol. Es hoja si no tiene atributo de división."""
    def __init__(self, pred, n, dist):
        self.pred = pred          # clase mayoritaria (índice)
        self.n = n                # cantidad de ejemplos que llegaron al nodo
        self.dist = dist          # conteo por clase
        self.atributo = None      # índice de columna de la división
        self.umbral = None        # None si el atributo es categórico
        self.hijos = {}           # {"<=": nodo, ">": nodo} o {valor_categoría: nodo}
        self.defecto = None       # hijo para faltantes / categorías no vistas

    @property
    def es_hoja(self):
        return self.atributo is None


class ArbolDecision:
    # color de cada clase en los gráficos (verde = sano ... rojo oscuro = enfermedad más grave)
    COLORES = ["#a6d96a", "#fee08b", "#fdae61", "#f46d43", "#a50026"]

    def __init__(self, criterio="razon_ganancia", max_profundidad=None,
                 min_muestras_split=2, min_muestras_hoja=1, categoricos=()):
        self.criterio = criterio                  # "razon_ganancia" | "ganancia"
        self.max_profundidad = max_profundidad
        self.min_muestras_split = min_muestras_split
        self.min_muestras_hoja = max(1, min_muestras_hoja)
        self.categoricos = set(categoricos)       # nombres de columnas categóricas

    # ---------- entrenamiento ----------
    def fit(self, X, y):
        self.nombres_ = list(X.columns) if hasattr(X, "columns") else [f"x{j}" for j in range(np.shape(X)[1])]
        X = np.asarray(X, dtype=float)
        self.es_cat_ = np.array([n in self.categoricos for n in self.nombres_])
        self.clases_, yi = np.unique(np.asarray(y), return_inverse=True)
        self.k_ = len(self.clases_)
        self.raiz_ = self._construir(X, yi, 0)
        return self

    def _construir(self, X, y, prof):
        conteos = np.bincount(y, minlength=self.k_)
        nodo = Nodo(int(conteos.argmax()), len(y), conteos)
        if (len(y) < self.min_muestras_split or conteos.max() == len(y)
                or (self.max_profundidad is not None and prof >= self.max_profundidad)):
            return nodo
        mejor = self._mejor_division(X, y)
        if mejor is None:
            return nodo
        j, umbral = mejor
        v = X[:, j]
        conocido = ~np.isnan(v)
        if umbral is not None:
            ramas = {"<=": conocido & (v <= umbral), ">": conocido & (v > umbral)}
        else:
            ramas = {c: v == c for c in np.unique(v[conocido])}
        clave_def = max(ramas, key=lambda c: ramas[c].sum())
        ramas[clave_def] = ramas[clave_def] | ~conocido      # faltantes -> rama más poblada
        nodo.atributo, nodo.umbral = j, umbral
        for c, m in ramas.items():
            nodo.hijos[c] = self._construir(X[m], y[m], prof + 1)
        nodo.defecto = nodo.hijos[clave_def]
        return nodo

    def _mejor_division(self, X, y):
        cands = []                                    # (ganancia, info_división, columna, umbral)
        for j in range(X.shape[1]):
            r = self._evaluar(X[:, j], y, j)
            if r is not None and r[0] > 1e-9:
                cands.append((r[0], r[1], j, r[2]))
        if not cands:
            return None
        if self.criterio == "ganancia":
            g, _, j, u = max(cands, key=lambda c: c[0])
        else:   # razón de ganancia, entre los atributos con ganancia >= promedio (C4.5)
            prom = np.mean([c[0] for c in cands])
            g, _, j, u = max((c for c in cands if c[0] >= prom - 1e-12),
                             key=lambda c: c[0] / c[1])
        return j, u

    def _evaluar(self, v, y, j):
        """Devuelve (ganancia, info_división, umbral) del atributo j, o None si no divide."""
        conocido = ~np.isnan(v)
        nk = int(conocido.sum())
        if nk < 2:
            return None
        v, yk = v[conocido], y[conocido]
        F = nk / len(y)                               # fracción de filas con valor conocido
        h0 = _entropia(np.bincount(yk, minlength=self.k_))
        if self.es_cat_[j]:
            cats, inv = np.unique(v, return_inverse=True)
            if len(cats) < 2:
                return None
            C = np.zeros((len(cats), self.k_))
            np.add.at(C, (inv, yk), 1)
            tam = C.sum(axis=1)
            ganancia = F * (h0 - (tam / nk * _entropia(C)).sum())
            p = tam / nk
            return ganancia, -(p * np.log2(p)).sum(), None
        orden = np.argsort(v, kind="stable")
        vs, ys = v[orden], yk[orden]
        cum = np.cumsum(np.eye(self.k_)[ys], axis=0)
        idx = np.where(vs[:-1] != vs[1:])[0]          # cortes posibles entre valores distintos
        if len(idx) == 0:
            return None
        L = cum[idx]
        R = cum[-1] - L
        nL = L.sum(axis=1)
        nR = nk - nL
        ok = (nL >= self.min_muestras_hoja) & (nR >= self.min_muestras_hoja)
        if not ok.any():
            return None
        idx, L, R, nL, nR = idx[ok], L[ok], R[ok], nL[ok], nR[ok]
        g = h0 - (nL * _entropia(L) + nR * _entropia(R)) / nk
        b = int(g.argmax())
        umbral = (vs[idx[b]] + vs[idx[b] + 1]) / 2
        pL = nL[b] / nk
        return F * g[b], -(pL * np.log2(pL) + (1 - pL) * np.log2(1 - pL)), umbral

    # ---------- predicción ----------
    def predict(self, X):
        X = np.asarray(X, dtype=float)
        out = np.zeros(len(X), dtype=int)
        self._predecir(self.raiz_, X, np.arange(len(X)), out)
        return self.clases_[out]

    def _predecir(self, nodo, X, idx, out):
        if len(idx) == 0:
            return
        if nodo.es_hoja:
            out[idx] = nodo.pred
            return
        v = X[idx, nodo.atributo]
        if nodo.umbral is not None:
            masks = {"<=": v <= nodo.umbral, ">": v > nodo.umbral}
        else:
            masks = {c: v == c for c in nodo.hijos}
        asignado = np.zeros(len(idx), dtype=bool)
        for c, m in masks.items():
            self._predecir(nodo.hijos[c], X, idx[m], out)
            asignado |= m
        if (~asignado).any():                          # faltante o categoría no vista
            self._predecir(nodo.defecto, X, idx[~asignado], out)

    # ---------- utilidades ----------
    def profundidad(self, nodo=None):
        nodo = nodo or self.raiz_
        return 0 if nodo.es_hoja else 1 + max(self.profundidad(h) for h in nodo.hijos.values())

    def n_hojas(self, nodo=None):
        nodo = nodo or self.raiz_
        return 1 if nodo.es_hoja else sum(self.n_hojas(h) for h in nodo.hijos.values())

    def a_dot(self, max_prof=3):
        """Devuelve el árbol en formato DOT (graphviz), mostrando hasta max_prof niveles.
        Nodo interno = atributo por el que se divide; hoja = clase mayoritaria (coloreada).
        n = ejemplos de entrenamiento que llegaron al nodo, [..] = cuántos hay de cada clase."""
        lineas = ["digraph Arbol {",
                  'node [shape=box, style="rounded,filled", fontname="Helvetica", fontsize=10];',
                  'edge [fontname="Helvetica", fontsize=10];']
        contador = [0]

        def rec(nodo, prof):
            i = contador[0]
            contador[0] += 1
            dist = " / ".join(str(int(x)) for x in nodo.dist)
            cortado = (not nodo.es_hoja) and prof >= max_prof
            if nodo.es_hoja or cortado:
                color = self.COLORES[nodo.pred % len(self.COLORES)]
                extra = "\\n(sigue...)" if cortado else ""
                etiqueta = f"clase {self.clases_[nodo.pred]}\\nn={nodo.n}\\n[{dist}]{extra}"
                lineas.append(f'n{i} [label="{etiqueta}", fillcolor="{color}"];')
                return i
            etiqueta = f"{self.nombres_[nodo.atributo]}\\nn={nodo.n}\\n[{dist}]"
            lineas.append(f'n{i} [label="{etiqueta}", fillcolor="#f2f2f2"];')
            for c, h in nodo.hijos.items():
                j = rec(h, prof + 1)
                cond = f"{c} {nodo.umbral:.2f}" if nodo.umbral is not None else f"= {c:g}"
                lineas.append(f'n{i} -> n{j} [label="{cond}"];')
            return i

        rec(self.raiz_, 0)
        lineas.append("}")
        return "\n".join(lineas)

    def imprimir(self, max_prof=3):
        """Muestra el árbol en texto hasta la profundidad indicada."""
        def rec(nodo, pref, prof):
            if nodo.es_hoja or prof >= max_prof:
                print(f"{pref}-> clase {self.clases_[nodo.pred]}  (n={nodo.n}, {nodo.dist.astype(int).tolist()})")
                return
            for c, h in nodo.hijos.items():
                nom = self.nombres_[nodo.atributo]
                cond = f"{nom} {c} {nodo.umbral:.2f}" if nodo.umbral is not None else f"{nom} = {c:g}"
                print(f"{pref}{cond}")
                rec(h, pref + "    ", prof + 1)
        rec(self.raiz_, "", 0)

## 5. Random Forest (scikit-learn)

`RandomForestClassifier` dentro de un *pipeline* con imputación por mediana (sklearn no acepta `NaN` en esta versión del modelo).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer


def crear_rf(n_arboles=100, max_profundidad=None, criterio="entropy", max_atributos="sqrt"):
    """Random Forest con imputación por mediana."""
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=n_arboles, max_depth=max_profundidad,
                               criterion=criterio, max_features=max_atributos,
                               random_state=SEED, n_jobs=-1))

## 6. Evaluación y comparación

Métricas: accuracy y precision / recall / F1 con promedio **macro** (cada clase pesa igual, lo que importa por el desbalance), más la matriz de confusión y los tiempos de entrenamiento y predicción.

In [ ]:
import time


def metricas(y_real, y_pred):
    """Métricas globales (precision/recall/F1 en promedio macro)."""
    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_real, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_real, y_pred, average="macro", zero_division=0),
    }


def evaluar(modelo, nombre):
    """Entrena, predice sobre prueba y devuelve (fila_de_resultados, y_pred)."""
    t0 = time.perf_counter(); modelo.fit(X_train, y_train); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); y_pred = modelo.predict(X_test); t_pred = time.perf_counter() - t0
    fila = {"modelo": nombre, **metricas(y_test, y_pred), "t_entren (s)": t_fit, "t_pred (s)": t_pred}
    return fila, y_pred


def matriz_confusion(y_pred):
    etiquetas = np.unique(y)
    return pd.DataFrame(confusion_matrix(y_test, y_pred, labels=etiquetas),
                        index=[f"real {e}" for e in etiquetas],
                        columns=[f"pred {e}" for e in etiquetas])

In [ ]:
dt = ArbolDecision(criterio="razon_ganancia", max_profundidad=None, categoricos=CATEGORICOS)
rf = crear_rf(n_arboles=100)

fila_dt, pred_dt = evaluar(dt, "DT propio (gain ratio)")
fila_rf, pred_rf = evaluar(rf, "Random Forest (100 árboles)")

display(pd.DataFrame([fila_dt, fila_rf]).set_index("modelo").round(4))
print(f"Árbol: profundidad {dt.profundidad()}, {dt.n_hojas()} hojas")
print("\nMatriz de confusión - DT propio")
display(matriz_confusion(pred_dt))
print("Matriz de confusión - Random Forest")
display(matriz_confusion(pred_rf))

In [ ]:
# Primeros niveles del árbol aprendido
dt.imprimir(max_prof=3)

## 7. Resultados y notas para el informe

TODO: registrar acá las simulaciones realizadas y los problemas encontrados.